In [ ]:
import sys
from contextlib import redirect_stdout 
import io

sys.path.append('../')

In [ ]:
import pandas as pd

df = pd.read_parquet('../ssi_all_features.parquet')
df.shape

In [ ]:
tmp = pd.read_csv('../../../../analytics_dbt/seeds/icd_10_to_cmqcc.csv')
diabetes_dx_codes = tmp.loc[tmp['category'].str.contains('diabetes'), 'dx'].tolist()
diabetes_dx_codes = [code.replace('.', '_') for code in diabetes_dx_codes]
diabetes_dx_codes

In [ ]:
dx_features = df.loc[df['description'] == 'diagnosis', 'feature_name'].unique().tolist()
glucose_features = df.loc[df['description'] == 'glucose', 'feature_name'].unique().tolist()
diabetes_dx_features = [x for x in dx_features if x.endswith('_has_dx') and (any(x.startswith(code) for code in diabetes_dx_codes))]
print(len(glucose_features), len(diabetes_dx_features))
diabetes_dx_features

In [ ]:
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
df_pivoted.columns.name = ''
df_pivoted

In [ ]:
from preprocess import preprocess

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_preprocessed = preprocess(df_pivoted)

In [ ]:
df_pivoted_preprocessed.isna().sum().sort_values(ascending=False)

In [ ]:
from feature_filter import feature_filter

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_filtered = feature_filter(df_pivoted_preprocessed, skip_medadmin_ingredient=True)

del df_pivoted_preprocessed

In [ ]:
df_pivoted_filtered.isna().sum().sort_values(ascending=False)

In [ ]:
glucose_features = [x for x in glucose_features if x in df_pivoted_filtered.columns]
diabetes_dx_features = [x for x in diabetes_dx_features if x in df_pivoted_filtered.columns]
len(glucose_features), len(diabetes_dx_features)

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

clf = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42),
                   param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]},
                  #  param_grid= = {'min_samples_split': [500]} 
                   verbose=3,
                   cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   scoring=['f1', 'roc_auc'],
                   refit='roc_auc')

### 1 baseline model, just everything

In [ ]:
sys.path.insert(0, '../../')

from src.model_training import nested_cv_evaluation_feature_selection

In [ ]:
pcornet_features = [x for x in df_pivoted_filtered.columns \
                    if x != 'target_variable' and x != 'birthid' and x != 'mother_patid']
all_features = []
for x in pcornet_features:
    if 'medadmin_ingredient' in x:
        if 'has_medadmin_ingredient' in x or 'earliest_minute' in x:
            all_features.append(x)
    else:
        all_features.append(x)

print(len(all_features))

In [ ]:
X = df_pivoted_filtered[all_features]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_baseline = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                          outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
metrics_baseline['roc_auc'].mean()

In [ ]:
import shap

def get_shap(clf, X, max_display=30):
    explainer = shap.TreeExplainer(clf, data=X,
                                   model_output='probability', feature_perturbation='interventional')
    shap_values = explainer(X, check_additivity=True)
    shap_values.values = shap_values.values[:,:,1]
    shap.plots.beeswarm(shap_values, max_display=max_display)
    return shap_values

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf, X, max_display=50)

### 2. remove diabetes diagnosis features

In [ ]:
X = df_pivoted_filtered[all_features].drop(columns=diabetes_dx_features)
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_2 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf, X, max_display=50)

### 3. remove glucose features

In [ ]:
X = df_pivoted_filtered[all_features].drop(columns=glucose_features)
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_3 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf, X, max_display=50)

### 4 remove glucose and diabetes features

In [ ]:
X = df_pivoted_filtered[all_features].drop(columns=diabetes_dx_features + glucose_features)
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_4 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

metrics_baseline['test'] = 'all features'
metrics_2['test'] = 'all features - diabetes_dx_features'
metrics_3['test'] = 'all features - glucose_features'
metrics_4['test'] = 'all features - diabetes_dx_features - glucose_features'
all_metrics = pd.concat([metrics_baseline, metrics_2, metrics_3, metrics_4], axis=0)

sns.barplot(all_metrics, x='roc_auc', y='test')
plt.xlim([0.65, 0.72])

In [ ]:
all_metrics.to_csv('check_diabetes_for_paper.csv', index=False)

In [ ]:
all_metrics.groupby('test')['roc_auc'].mean()

In [ ]:
from scipy.stats import ttest_rel

a = all_metrics.loc[all_metrics['test'] == 'all features', 'roc_auc']
b = all_metrics.loc[all_metrics['test'] == 'all features - diabetes_dx_features - glucose_features', 'roc_auc']

ttest_rel(a, b)